# Augmentations-2: Augmentations

In [ ]:
import os
import time
from pathlib import Path

root = Path.cwd()
print(root)
if root.name == "ipynb":
    root = root.parent
    os.chdir(root)
print(root)

import zarr
import tifffile
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
import ipywidgets as widgets
from scipy.ndimage import rotate

# Tif
NO_AUG_DIR = root / f"fisbe/biapy-no-aug/test/raw"
CHANNEL_ROT_DIR = root / f"fisbe/biapy-channel-rot"
CHANNEL_COLOR_DIR = root / f"fisbe/biapy-channel-color-test-val"
# Zarr
ORIGINAL_DIR = Path("fisbe/completely/test")
# PNG
ORIGINAL_MIP_DIR = Path("fisbe/mips/completely/test")


## Helper Functions

### Imaging

In [ ]:
def gen_mip(raw, axis) -> np.ndarray:
    """Max-intensity projection of RGB channels. For CZYX: axis=1 is the z projection."""
    mip = raw.astype(np.float32).max(axis=axis)
    mip = (mip - mip.min()) / (np.ptp(mip) + 1e-8)
    return np.moveaxis(mip, 0, -1)  # (Y, X, C)

In [ ]:
PCT_LOW, PCT_HIGH, GAMMA = 1.0, 99.5, 0.72
def enhance_display(data: np.ndarray) -> np.ndarray:
    """Shared contrast + gamma; same defaults as web/server/services/volume_pipeline.py."""
    v = data.astype(np.float32)
    if not np.any(v > 0):
        return np.zeros_like(v)
    sample = v[v > 0] if np.count_nonzero(v) > 256 else v.ravel()
    lo, hi = np.percentile(sample, [PCT_LOW, PCT_HIGH])
    if hi <= lo:
        lo, hi = float(v.min()), float(v.max())
    if hi <= lo:
        return np.zeros_like(v)
    return np.clip((v - lo) / (hi - lo), 0, 1) ** GAMMA
def fisbe_rgb_mip(raw_czyx, z_axis=1) -> np.ndarray:
    """raw: CZYX. Returns Y×X×3 uint8, web-matched vibrance."""
    c = min(raw_czyx.shape[0], 3)
    mip_cyx = np.asarray(raw_czyx[:c], dtype=np.float32).max(axis=z_axis)  # C,Y,X
    rgb01 = enhance_display(mip_cyx)  # shared stretch across channels (keeps hue)
    return np.moveaxis((rgb01 * 255).astype(np.uint8), 0, -1)

In [ ]:
def mip_fisbe_gt_instance(labels, sample_name, z_axis=1):
    arr = np.asarray(labels)
    mip = arr.max(axis=z_axis)       # (C, Y, X)
    lab2d = mip.max(axis=0)          # (Y, X)

    rgb = np.zeros((*lab2d.shape, 3), dtype=np.float32)
    for lab in np.unique(lab2d):
        if lab == 0:
            continue
        color = np.random.randint(72, 255, 3).astype(np.float32)
        rgb[lab2d == lab] = color
    rgb /= 255.0

    fig, ax = plt.subplots(1, 1, figsize=(2, 2), dpi=100)
    ax.imshow(rgb)
    ax.set_title(f"GT instance MIP {sample_name}")
    ax.axis("off")
    plt.show()


In [ ]:
def mip_biapy_gt_instance(labels, sample_name, z_axis=0):
    arr = np.asarray(labels)
    mip = arr.max(axis=z_axis)       # (Y, X)

    rgb = np.zeros((*mip.shape, 3), dtype=np.float32)
    for lab in np.unique(mip):
        if lab == 0:
            continue
        color = np.random.randint(72, 255, 3).astype(np.float32)
        rgb[mip == lab] = color
    rgb /= 255.0

    fig, ax = plt.subplots(1, 1, figsize=(8, 8), dpi=100)
    ax.imshow(rgb)
    ax.set_title(f"GT instance MIP {sample_name}")
    ax.axis("off")
    plt.show()


### Stats

In [ ]:
def per_channel_stats(data: np.ndarray, axis: int = 0) -> pd.DataFrame:
    """Basic stats of values across channels (axis 0 for CZYX).

    Expects shape like (C, Z, Y, X), e.g. (3, 390, 680, 680) uint16.
    """
    data = np.asarray(data)
    n_ch = data.shape[axis]
    rows = []
    for c in range(n_ch):
        ch = np.take(data, c, axis=axis).ravel()
        nonzero = ch[ch > 0]
        rows.append({
            "channel": c,
            "dtype": str(data.dtype),
            "n": ch.size,
            "n_nonzero": int(nonzero.size),
            "min": int(ch.min()) if ch.size else np.nan,
            "max": int(ch.max()) if ch.size else np.nan,
            "mean": float(ch.mean()) if ch.size else np.nan,
            "std": float(ch.std()) if ch.size else np.nan,
            "median": float(np.median(ch)) if ch.size else np.nan,
            "p1": float(np.percentile(ch, 1)) if ch.size else np.nan,
            "p99": float(np.percentile(ch, 99)) if ch.size else np.nan,
            "mean_nonzero": float(nonzero.mean()) if nonzero.size else 0.0,
            "median_nonzero": float(np.median(nonzero)) if nonzero.size else 0.0,
        })
    return pd.DataFrame(rows).set_index("channel")


def per_channel_histogram(
    data: np.ndarray,
    sample_name: str,
    axis: int = 0,
    bins: int = 256,
    range_=None,
    skip_zeros: bool = True,
    log_y: bool = True,
    figsize=(12, 3.5),
):
    """Basic histogram of value distribution across channels (axis 0 for CZYX).

    Plots one subplot per channel. Returns (fig, axes, counts).
    """
    data = np.asarray(data)
    n_ch = data.shape[axis]
    if range_ is None:
        range_ = (int(data.min()), int(data.max()) + 1)

    fig, axes = plt.subplots(1, n_ch, figsize=figsize, sharey=True)
    if n_ch == 1:
        axes = [axes]

    counts = []
    for c, ax in enumerate(axes):
        ch = np.take(data, c, axis=axis).ravel()
        if skip_zeros:
            ch = ch[ch > 0]
        hist, edges = np.histogram(ch, bins=bins, range=range_)
        counts.append(hist)
        centers = (edges[:-1] + edges[1:]) / 2
        ax.bar(centers, hist, width=np.diff(edges), align="center", alpha=0.8)
        ax.set_title(f"channel {c}")
        ax.set_xlabel("value")
        if log_y:
            ax.set_yscale("log")
        ax.set_xlim(range_)

    axes[0].set_ylabel("count" + (" (log)" if log_y else ""))
    fig.suptitle(f"Per-channel value histogram {sample_name}", y=1.02)
    fig.tight_layout()
    return fig, axes, counts


## Base Image

### BiaPy Pre - Processing

In [ ]:
SAMPLE_ID = "JRC_SS04989-20160318_24_A2"
path = NO_AUG_DIR / f"{SAMPLE_ID}.tif"
img = tifffile.imread(path)
img = np.moveaxis(img, 0, 1)  # BiaPy ZCYX -> CZYX
print(path)
print(img.shape, img.dtype)

In [ ]:
plt.figure(figsize=(6, 6))
plt.imshow(gen_mip(img, 1))
plt.axis("off")
plt.title(f"axis-aligned MIP: {path.name}")
plt.show()

### Rotation Augmentations

In [ ]:
for p in list(CHANNEL_ROT_DIR.glob('*.tif'))[:4]:
    sample = p.stem
    raw = tifffile.imread(p, mode="r")
    raw = raw.swapaxes(0, 1)
    print(raw.shape)
    rgb = fisbe_rgb_mip(raw)

    fig, ax = plt.subplots(1, 1, figsize=(20, 6))
    ax.imshow(rgb)
    ax.axis("off")
    ax.set_title(f"{p}")
    # display(Image(filename=(ORIGINAL_MIP_DIR / f'{sample}.png').as_posix()), height=100)
    plt.show()

### Color Augmentations

In [ ]:
# for p in list(CHANNEL_ROT_DIR.glob('*.tif'))[:4]:
for p in list(CHANNEL_COLOR_DIR.glob('*.tif'))[:4]:
    sample = p.stem
    raw = tifffile.imread(p, mode="r")
    raw = raw.swapaxes(0, 1)
    print(raw.shape)
    rgb = fisbe_rgb_mip(raw)

    fig, ax = plt.subplots(1, 1, figsize=(20, 6))
    ax.imshow(rgb)
    ax.axis("off")
    ax.set_title(f"{p}")
    # display(Image(filename=(ORIGINAL_MIP_DIR / f'{sample}.png').as_posix()), height=100)
    plt.show()

## Stats

In [ ]:
total_stats = []
for p in ORIGINAL_DIR.glob('*.zarr'):
    sample = p.stem
    raw = zarr.open(p, mode="r", path="volumes/raw")
    total_stats.append(per_channel_stats(raw))
total_stats= pd.concat(total_stats)
total_stats

In [ ]:
for p in ORIGINAL_DIR.glob('*.zarr'):
    sample = p.stem
    raw = zarr.open(p, mode="r", path="volumes/raw")
    per_channel_histogram(data=raw, sample_name=sample)

## Original GT instances

In [ ]:
# for p in ORIGINAL_DIR.glob('*.zarr'):
#     sample = p.stem
#     labels = zarr.open(p, mode="r", path="volumes/gt_instances")
#     print(labels.shape)
#     mip_fisbe_gt_instance(labels, sample)

## Applied Augmentation - Rotation and Channel Flip

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

AUG_DATA_DIR = root / "fisbe/biapy-debug/test/raw"
sample_augmentations = sorted(AUG_DATA_DIR.glob("JRC_SS04989-20160318_24_A2*"))
print(len(sample_augmentations))


def load_mip(path: Path):
    img = tifffile.imread(path)
    img = np.moveaxis(img, 0, 1)  # BiaPy ZCYX -> CZYX
    return path, fisbe_rgb_mip(img, 1)


mips = []
with ThreadPoolExecutor(max_workers=2) as ex:
    futures = {ex.submit(load_mip, p): i for i, p in enumerate(sample_augmentations)}
    for fut in tqdm(as_completed(futures), total=len(futures)):
        i = futures[fut]
        mips.append(fut.result())

for chunk_start in range(0, len(mips), 4):
    chunk = mips[chunk_start : chunk_start + 4]
    fig, axes = plt.subplots(2, 2, figsize=(12, 12), dpi=150)
    for i, (path, mip) in enumerate(chunk):
        ax = axes[i // 2, i % 2]
        ax.imshow(mip)
        ax.set_title(path.stem.replace("R38F04-20181005_63_G3", ""), fontsize=8)
        ax.axis("off")
    plt.tight_layout(pad=0.2, w_pad=0.1, h_pad=0.2)
    plt.show()

